# Item Features

Converts the cleaned product metadata into numeric item features for the recommendation models.

Three text representations get built. BLaIR is the primary encoder because it was pre-trained on Amazon Reviews 2023 item metadata. MiniLM and TF-IDF+SVD are included as comparison baselines: MiniLM evaluates a general-purpose encoder, while TF-IDF+SVD provides a traditional bag-of-words baseline. Only BLaIR is used in the main experiments.

Features are generated once for the full dataset and then sliced into each subset using the item mappings from Notebook 05, ensuring the feature rows match each subset's local item indices.

In [3]:
import pyarrow.parquet as pq
from pathlib import Path

In [4]:


METADATA_PATH = Path("../data/processed/encoded_earliest/metadata_encoded.parquet")

md = pq.read_table(METADATA_PATH, columns=["details_text"]).to_pandas()
print(f"{md.details_text.str.contains('Best Sellers Rank', na=False).mean():.1%} of "
      f"{len(md):,} items")

56.3% of 368,228 items


In [5]:
md.head()

,details_text
0,Dimensions: 6 x 0.57 x 6 inches | Hardcover: 1...
1,Dimensions: 7.5 x 1 x 9.3 inches | ISBN 10: 00...
2,Dimensions: 5.51 x 0.51 x 8.34 inches | ISBN 1...
3,Dimensions: 6.02 x 1.61 x 9.13 inches | ISBN 1...
4,Dimensions: 7.9 x 0.9 x 10.1 inches | Hardcove...


In [6]:
matches = md[md.details_text.str.contains("Best Sellers Rank", na=False)]

print(matches.iloc[0]["details_text"])

Batteries: 2 AAA batteries required. | Best Sellers Rank: {'Electronics': 235167, 'Remote Controls': 12947} | Brand: URC | Connectivity Technology: Infrared | Controller Type: Button Control | Date First Available: January 31, 2013 | Is Discontinued By Manufacturer: No | Item Weight: 0.8 ounces | Item model number: UR5U-8780L | Manufacturer: Universal Remote Control, Inc. | Max Number of Supported Devices: 5 | Package Dimensions: 8.6 x 3.3 x 1 inches | Special Feature: Universal


## 1. Imports and Configuration

In [10]:
%load_ext autoreload
%autoreload 2

import json
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Paths
ENCODED_DIR = PROJECT_ROOT / "data" / "processed" / "encoded_earliest"
METADATA_PATH = ENCODED_DIR / "metadata_encoded.parquet"
SUBSETS_DIR = PROJECT_ROOT / "data" / "processed" / "subsets_earliest"
FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features_earliest"
FIGURES_DIR = PROJECT_ROOT / "figures"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Config
EXPECTED_ITEMS = 368_228
SUBSET_LEVELS = ["small", "medium", "full"]

BLAIR_MODEL = "hyp1231/blair-roberta-base"      # primary
MINILM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # control
SVD_COMPONENTS = 256                             # control, TF-IDF reduced to this

MAX_TOKENS = 512          # metadata text is long
ENCODE_BATCH = 64
SEED = 42

# stripped before encoding: popularity measured at crawl time in 2023, which is
# after the test period starts under the timestamp protocol
LEAKY_DETAIL_KEYS = ["Best Sellers Rank"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"device: {DEVICE}")
print(f"primary encoder : {BLAIR_MODEL}")
print(f"controls        : {MINILM_MODEL}, TF-IDF+SVD({SVD_COMPONENTS})")
print(f"stripping       : {LEAKY_DETAIL_KEYS}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
device: cuda
primary encoder : hyp1231/blair-roberta-base
controls        : sentence-transformers/all-MiniLM-L6-v2, TF-IDF+SVD(256)
stripping       : ['Best Sellers Rank']


## 2. Load metadata and strip post-cutoff fields

The metadata was collected in 2023, while the timestamp evaluation predicts interactions from July 2022 onward. To avoid temporal leakage, fields containing information that was only known after the evaluation period are removed before encoding.

`Best Sellers Rank` is removed because it reflects product popularity measured when the metadata was collected, providing future information to the model. `Date First Available` is retained because a product's release date was already known at prediction time and can help with cold-start recommendations.

This step is performed here rather than in Notebook 03 so the original cleaned metadata remains unchanged and the feature preparation process is fully documented.

In [4]:
metadata = pq.read_table(
    METADATA_PATH,
    columns=["item_idx", "title", "store", "main_category",
             "details_text", "combined_metadata_text"],
).to_pandas()

print(f"{len(metadata):,} items")
assert len(metadata) == EXPECTED_ITEMS
assert metadata["item_idx"].is_unique
assert metadata["item_idx"].min() == 0
assert metadata["item_idx"].max() == EXPECTED_ITEMS - 1

# keep row order aligned with item_idx so slicing by index is safe later
metadata = metadata.sort_values("item_idx").reset_index(drop=True)
assert (metadata["item_idx"].to_numpy() == np.arange(len(metadata))).all()

metadata.head(2)

368,228 items


,item_idx,title,store,main_category,details_text,combined_metadata_text
0,0,Strange Planet (Strange Planet Series),Nathan W. Pyle (Author),Books,Dimensions: 6 x 0.57 x 6 inches | Hardcover: 1...,Strange Planet (Strange Planet Series) Nathan ...
1,1,High-Power Audio Amplifier Construction Manual,G. Randy Slone (Author),Books,Dimensions: 7.5 x 1 x 9.3 inches | ISBN 10: 00...,High-Power Audio Amplifier Construction Manual...


In [6]:
from src import features as ft

watch = LEAKY_DETAIL_KEYS + ["Date First Available"]
before = {k: metadata["combined_metadata_text"].str.contains(k, na=False).mean()
          for k in watch}

t0 = time.time()
metadata["clean_text"], missing = ft.strip_leaky_from_combined(
    metadata["combined_metadata_text"], metadata["details_text"], LEAKY_DETAIL_KEYS
)
print(f"stripped in {time.time() - t0:.0f}s, {missing} rows where details "
      f"could not be located in the combined text\n")

after = {k: metadata["clean_text"].str.contains(k, na=False).mean() for k in watch}
for key in watch:
    print(f"{key:>22}: {before[key]:6.1%} -> {after[key]:6.1%}")

assert missing == 0, "details_text did not appear verbatim in some combined rows"

# any residual is text that mentions the phrase outside the details field, e.g. a
# seller writing it in a description - worth seeing before deciding what to do
for key in LEAKY_DETAIL_KEYS:
    residual = metadata[metadata["clean_text"].str.contains(key, na=False)]
    print(f"\nresidual '{key}': {len(residual):,} items")
    if len(residual):
        print(residual["clean_text"].iloc[0][-200:])

stripped in 5s, 0 rows where details could not be located in the combined text

     Best Sellers Rank:  56.3% ->   0.0%
  Date First Available:  94.2% ->  94.2%

residual 'Best Sellers Rank': 0 items


In [ ]:
# check the edit is surgical (only the leaky segment should be gone)
example = metadata[metadata["combined_metadata_text"].str.contains("Best Sellers Rank")].iloc[0]
print("BEFORE:\n", example["combined_metadata_text"][-320:], "\n")
print("AFTER:\n", example["clean_text"][-320:], "\n")
print(f"characters removed: {len(example['combined_metadata_text']) - len(example['clean_text'])}")

stats_before = ft.text_length_stats(metadata["combined_metadata_text"])
stats_after = ft.text_length_stats(metadata["clean_text"])

pd.DataFrame({"before": stats_before, "after": stats_after}).round(1)

BEFORE:
 roller Type: Button Control | Date First Available: January 31, 2013 | Is Discontinued By Manufacturer: No | Item Weight: 0.8 ounces | Item model number: UR5U-8780L | Manufacturer: Universal Remote Control, Inc. | Max Number of Supported Devices: 5 | Package Dimensions: 8.6 x 3.3 x 1 inches | Special Feature: Universal 

AFTER:
 roller Type: Button Control | Date First Available: January 31, 2013 | Is Discontinued By Manufacturer: No | Item Weight: 0.8 ounces | Item model number: UR5U-8780L | Manufacturer: Universal Remote Control, Inc. | Max Number of Supported Devices: 5 | Package Dimensions: 8.6 x 3.3 x 1 inches | Special Feature: Universal 

characters removed: 71


,before,after
mean_chars,1845.7,1809.0
median_chars,1606.0,1563.0
p95_chars,3814.0,3768.0
mean_words,297.6,292.3
median_words,258.0,252.0
p95_words,619.0,612.0
max_words,15452.0,15447.0


## 3. Text encoding with BLaIR

BLaIR is the primary encoder because it was pre-trained by the dataset authors on Amazon Reviews 2023 item metadata, matching the data used in this project.

RoBERTa accepts a maximum of 512 tokens. Since the metadata varies in length, the token distribution is measured before selecting a truncation limit.

When truncation occurs, the beginning of the metadata is preserved, retaining fields such as the title, store, categories, and description while truncating the less informative details section.

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BLAIR_MODEL)

sample = metadata["clean_text"].sample(2000, random_state=SEED).tolist()
lengths = np.array([len(tokenizer.encode(t, truncation=False)) for t in sample])

print(f"tokens per item (sample of {len(sample):,})")
for q in [50, 75, 90, 95, 99]:
    print(f"  p{q}: {np.percentile(lengths, q):>6.0f}")
print(f"  max: {lengths.max():>6.0f}")

words = np.array([len(t.split()) for t in sample])
print(f"\ntokens per word: {(lengths / np.maximum(words, 1)).mean():.2f}")

for limit in [128, 256, 384, 512]:
    print(f"  at {limit:>3} tokens: {(lengths <= limit).mean():6.1%} of items complete")

c:\Projects\gnn-recommender-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Projects\gnn-recommender-dissertation\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\xhani\.cache\huggingface\hub\models--hyp1231--blair-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator

tokens per item (sample of 2,000)
  p50:    402
  p75:    572
  p90:    793
  p95:    982
  p99:   1574
  max:   9058

tokens per word: 1.57
  at 128 tokens:   5.7% of items complete
  at 256 tokens:  26.1% of items complete
  at 384 tokens:  48.0% of items complete
  at 512 tokens:  67.5% of items complete


In [11]:
from transformers import AutoModel

blair = AutoModel.from_pretrained(BLAIR_MODEL)

bench_texts = metadata["clean_text"].sample(2000, random_state=SEED).tolist()
t0 = time.time()
_ = ft.encode_blair(bench_texts, blair, tokenizer, DEVICE,
                    batch_size=ENCODE_BATCH, max_tokens=MAX_TOKENS, progress_every=0)
elapsed = time.time() - t0

rate = len(bench_texts) / elapsed
print(f"{len(bench_texts):,} items in {elapsed:.0f}s  ({rate:.0f} items/s)")
print(f"estimated full run: {len(metadata) / rate / 60:.0f} minutes for {len(metadata):,} items")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13595.25it/s]


2,000 items in 18s  (111 items/s)
estimated full run: 55 minutes for 368,228 items


In [ ]:
BLAIR_PATH = FEATURES_DIR / "blair_embeddings.npy"

if BLAIR_PATH.exists():
    blair_emb = np.load(BLAIR_PATH)
    print(f"loaded existing embeddings {blair_emb.shape}")
else:
    t0 = time.time()
    blair_emb = ft.encode_blair(
        metadata["clean_text"].tolist(),
        blair, tokenizer, DEVICE,
        batch_size=ENCODE_BATCH,
        max_tokens=MAX_TOKENS,
        progress_every=500,
    )
    print(f"\nencoded in {(time.time() - t0) / 60:.0f} minutes")

    # save so an hour of work cannot be lost downstream
    np.save(BLAIR_PATH, blair_emb)
    print(f"saved -> {BLAIR_PATH.relative_to(PROJECT_ROOT)} "
          f"({BLAIR_PATH.stat().st_size / 1e6:.0f} MB)")

blair_emb.shape, blair_emb.dtype

  500/5754 batches
  1000/5754 batches
  1500/5754 batches
  2000/5754 batches
  2500/5754 batches
  3000/5754 batches
  3500/5754 batches
  4000/5754 batches
  4500/5754 batches
  5000/5754 batches
  5500/5754 batches

encoded in 19 minutes
saved -> data\processed\features_earliest\blair_embeddings.npy (1131 MB)


((368228, 768), dtype('float32'))

Shape and norm checks verify that the embeddings are numerically valid. As a final sanity check, the nearest neighbours of a few example items are inspected using cosine similarity to confirm that similar products receive similar embeddings.

In [13]:
assert blair_emb.shape == (len(metadata), 768)
assert np.isfinite(blair_emb).all(), "non-finite values in embeddings"

norms = np.linalg.norm(blair_emb, axis=1)
print(f"L2 norms: min {norms.min():.4f}, max {norms.max():.4f}")
assert np.allclose(norms, 1.0, atol=1e-3), "embeddings are not unit length"

print(f"rows identical to their neighbour: "
      f"{int((blair_emb[1:] == blair_emb[:-1]).all(axis=1).sum())}")

L2 norms: min 1.0000, max 1.0000
rows identical to their neighbour: 73


In [14]:
emb_gpu = torch.from_numpy(blair_emb).to(DEVICE)
rng = np.random.default_rng(SEED)

for query in rng.choice(len(metadata), 4, replace=False):
    sims = emb_gpu @ emb_gpu[query]           # already unit length, so this is cosine
    top = torch.topk(sims, 6).indices.cpu().numpy()

    print(f"QUERY: {metadata.title.iloc[query][:80]}")
    for rank, j in enumerate(top[1:], 1):     # skip the item itself
        print(f"   {rank}. [{sims[j]:.3f}] {metadata.title.iloc[j][:75]}")
    print()

del emb_gpu
torch.cuda.empty_cache()

QUERY: TeckNet 3 Port HDMI Auto Switch Box Plus Remote - 3x1 HUB (3 Way Input 1 Output)
   1. [0.974] 4 Port HDMI Switch, HDMI Switch 4 in 1 Out, HDMI Switch Remote, HDMI Switch
   2. [0.973] TeckNet 3 Port Auto Switch Box plus Remote - 3x1 HUB (3 way input 1 output)
   3. [0.972] HDMI Switch Support HDCP 1080p VAlinks® Mini 3-Port Intelligent 4K HDMI Aut
   4. [0.971] HDMI Switcher 4 Port, HDMI Switch with IR Remote, Support 4k@30hz, 3D and A
   5. [0.970] PORTTA HDMI Switch 4K, HDMI Switcher 3 in 1 Out HDMI Splitter, HDMI Selecto

QUERY: VideoSecu TV LCD Monitor Flat Panel Wall Mount Compatible VESA 75X75 100X100 19"
   1. [0.986] VideoSecu TV Wall Mount Flush Ultra Slim Bracket for Most 27"-47" LCD LED F
   2. [0.985] VideoSecu LCD LED Monitor TV Wall Mount for 19" 20" 22" 23" 24" 26" 27" 30"
   3. [0.984] GForce GF-686-328 Low Profile Fixed Mount for 14"-32" TVs and Computer Moni
   4. [0.984] GForce GF-686-329 Low Profile Fixed TV Wall Mount Bracket for 10 to 32 inch
   5. [0.982]

In [15]:
sample_idx = rng.choice(len(blair_emb), 20_000, replace=False)
sample = blair_emb[sample_idx]

pairs = rng.choice(len(sample), (5000, 2))
raw_cos = (sample[pairs[:, 0]] * sample[pairs[:, 1]]).sum(1)

centred = sample - sample.mean(0)
centred /= np.linalg.norm(centred, axis=1, keepdims=True)
centred_cos = (centred[pairs[:, 0]] * centred[pairs[:, 1]]).sum(1)

print(f"mean cosine, raw     : {raw_cos.mean():.3f}  (sd {raw_cos.std():.3f})")
print(f"mean cosine, centred : {centred_cos.mean():.3f}  (sd {centred_cos.std():.3f})")

# how many directions actually carry the variation
sv = np.linalg.svd(sample - sample.mean(0), compute_uv=False)
var = np.cumsum(sv ** 2) / np.sum(sv ** 2)
for target in [0.90, 0.95, 0.99]:
    print(f"components for {target:.0%} variance: {int(np.searchsorted(var, target)) + 1} / 768")

mean cosine, raw     : 0.645  (sd 0.078)
mean cosine, centred : 0.003  (sd 0.158)
components for 90% variance: 96 / 768
components for 95% variance: 165 / 768
components for 99% variance: 436 / 768


In [16]:
dup_rows = np.flatnonzero((blair_emb[1:] == blair_emb[:-1]).all(axis=1))
metadata.loc[[dup_rows[0], dup_rows[0] + 1], ["item_idx", "title"]]

,item_idx,title
63,63,"Barnes & Noble NOOK Tablet 16gb (Color, BNTV250)"
64,64,"Barnes & Noble NOOK Tablet 16gb (Color, BNTV250)"


## 4. Control encoders

Two additional text encoders are included for comparison:

- **MiniLM** - evaluates whether domain-specific pre-training provides an advantage over a general-purpose sentence encoder.
- **TF-IDF + SVD** - provides a traditional bag-of-words baseline to compare against neural text representations.

These encoders are used in two places: the encoder comparison in Notebook 11, which asks on the main recommendation task whether domain-specific pre-training gives an advantage, and the cold-start evaluation, where item features are the primary source of information. Each model is used with its recommended input length to ensure a fair comparison.

In [29]:
from src import features as ft

In [18]:
from sentence_transformers import SentenceTransformer

MINILM_PATH = FEATURES_DIR / "minilm_embeddings.npy"

if MINILM_PATH.exists():
    minilm_emb = np.load(MINILM_PATH)
    print(f"loaded {minilm_emb.shape}")
else:
    minilm = SentenceTransformer(MINILM_MODEL, device=DEVICE)
    print(f"max_seq_length: {minilm.max_seq_length}")

    t0 = time.time()
    minilm_emb = minilm.encode(
        metadata["clean_text"].tolist(),
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    print(f"encoded in {(time.time() - t0) / 60:.1f} minutes")

    np.save(MINILM_PATH, minilm_emb)

print(minilm_emb.shape)
assert np.allclose(np.linalg.norm(minilm_emb, axis=1), 1.0, atol=1e-3)

c:\Projects\gnn-recommender-dissertation\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\xhani\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11440.42it/s]


max_seq_length: 256


Batches: 100%|██████████| 1439/1439 [07:13<00:00,  3.32it/s]


encoded in 7.3 minutes
(368228, 384)


In [19]:
TFIDF_PATH = FEATURES_DIR / "tfidf_svd_embeddings.npy"

if TFIDF_PATH.exists():
    tfidf_emb = np.load(TFIDF_PATH)
    print(f"loaded {tfidf_emb.shape}")
else:
    t0 = time.time()
    tfidf_emb, explained = ft.build_tfidf_svd(
        metadata["clean_text"].tolist(),
        n_components=SVD_COMPONENTS,
        seed=SEED,
    )
    print(f"built in {(time.time() - t0) / 60:.1f} minutes, "
          f"{explained:.1%} of TF-IDF variance retained")
    np.save(TFIDF_PATH, tfidf_emb)

print(tfidf_emb.shape)

built in 1.3 minutes, 26.4% of TF-IDF variance retained
(368228, 256)


In [20]:
# same queries as before, so the three encoders can be compared directly
encoders = {"BLaIR": blair_emb, "MiniLM": minilm_emb, "TF-IDF+SVD": tfidf_emb}

for query in rng.choice(len(metadata), 2, replace=False):
    print(f"QUERY: {metadata.title.iloc[query][:75]}\n")
    for name, emb in encoders.items():
        e = torch.from_numpy(emb).to(DEVICE)
        top = torch.topk(e @ e[query], 4).indices.cpu().numpy()[1:]
        print(f"  {name}:")
        for j in top:
            print(f"     {metadata.title.iloc[j][:68]}")
        del e
        torch.cuda.empty_cache()
    print()

QUERY: YOUPECK 11.6 11 inch Laptop Case, Laptop Shoulder Bag, Waterproof Notebook 

  BLaIR:
     YOUPECK 12 13 inch Laptop Case, Laptop Shoulder Bag, Waterproof Note
     YOUPECK 10 10.5 11 11.6 inch Laptop Case, Laptop Shoulder Bag, Water
     Laptop Bag,BRINCH Slim Water Resistant Laptop Messenger Bag Portable
  MiniLM:
     YOUPECK 12 13 inch Laptop Case, Laptop Shoulder Bag, Waterproof Note
     YOUPECK 10 10.5 11 11.6 inch Laptop Case, Laptop Shoulder Bag, Water
     Laptop Bag,BRINCH Slim Water Resistant Laptop Messenger Bag Portable
  TF-IDF+SVD:
     YOUPECK 10 10.5 11 11.6 inch Laptop Case, Laptop Shoulder Bag, Water
     YOUPECK 12 13 inch Laptop Case, Laptop Shoulder Bag, Waterproof Note
     Laptop Bag,BRINCH Slim Water Resistant Laptop Messenger Bag Portable

QUERY: Geekria Protein Leather Headband Pad Compatible with Bose QuietComfort 35 i

  BLaIR:
     Geekria Protein Leather Headband Pad Compatible with Bose QuietComfo
     Learsoon Replacement Top Headband Pad Cushio

## 5. Categorical features

`main_category` and `store` are already included in the text used by the encoders. An explicit categorical feature is therefore added only where it provides additional value.

`main_category` is represented as a one-hot feature because it has low cardinality and provides a clear categorical signal at little cost.

`store` is not included separately because it has very high cardinality and would add many dimensions while largely duplicating information already captured by the text embeddings.

In [21]:
print("cardinality")
print(f"  main_category: {metadata.main_category.nunique():,}")
print(f"  store        : {metadata.store.nunique():,}")

print("\nmain_category distribution")
print(metadata.main_category.value_counts().head(15).to_string())

top_share = metadata.store.value_counts().head(200).sum() / len(metadata)
print(f"\ntop 200 stores cover {top_share:.1%} of items")

cardinality
  main_category: 39
  store        : 49,344

main_category distribution
main_category
All Electronics              100424
Computers                     97946
Camera & Photo                52624
Cell Phones & Accessories     32833
Home Audio & Theater          27693
Industrial & Scientific       11700
nan                            6632
Car Electronics                6062
Tools & Home Improvement       5326
Office Products                4788
Amazon Home                    3380
Amazon Devices                 3373
Sports & Outdoors              2812
AMAZON FASHION                 2723
Automotive                     2555

top 200 stores cover 29.8% of items


In [22]:
# "nan" is a literal string left by the Notebook 03 missing-value handling.
# Missingness is informative so it stays as a category, just with a readable label.
categories = metadata["main_category"].replace("nan", "unknown").fillna("unknown")

category_levels = sorted(categories.unique())
category_index = {c: i for i, c in enumerate(category_levels)}

category_onehot = np.zeros((len(metadata), len(category_levels)), dtype=np.float32)
category_onehot[np.arange(len(metadata)), categories.map(category_index).to_numpy()] = 1.0

print(f"{len(category_levels)} categories -> one-hot {category_onehot.shape}")
print(f"items with unknown category: {int(category_onehot[:, category_index['unknown']].sum()):,}")

assert category_onehot.sum() == len(metadata)          # exactly one category per item
assert (category_onehot.sum(axis=1) == 1).all()

np.save(FEATURES_DIR / "category_onehot.npy", category_onehot)
with open(FEATURES_DIR / "category_levels.json", "w", encoding="utf-8") as fh:
    json.dump(category_levels, fh, indent=2)

39 categories -> one-hot (368228, 39)
items with unknown category: 6,632


## 6. Assemble the feature matrices

Each item's feature vector is formed by concatenating its text embedding with the one-hot encoded `main_category`.

The text embeddings are centred and re-normalised before concatenation to improve their conditioning, while the categorical features retain their original representation. Features are assembled once for the full dataset and then sliced into each subset using the item mappings, ensuring the feature rows match each subset's local item indices.

A final alignment check verifies that the global feature matrix has been mapped correctly to every subset before the features are used for model training.

In [24]:
feature_blocks = {}
for name, emb in [("blair", blair_emb), ("minilm", minilm_emb), ("tfidf_svd", tfidf_emb)]:
    feature_blocks[name] = ft.assemble_features(emb, category_onehot, centre=True)
    print(f"{name:>10}: {feature_blocks[name].shape}  "
          f"({feature_blocks[name].nbytes / 1e6:,.0f} MB)")

blair_features = feature_blocks["blair"]

     blair: (368228, 807)  (1,189 MB)
    minilm: (368228, 423)  (623 MB)
 tfidf_svd: (368228, 295)  (435 MB)


In [25]:
for name, feats in feature_blocks.items():
    text_dim = feats.shape[1] - category_onehot.shape[1]
    text_part, cat_part = feats[:, :text_dim], feats[:, text_dim:]

    print(f"{name}")
    print(f"  text block  : mean {text_part.mean():+.5f}, "
          f"row norm {np.linalg.norm(text_part, axis=1).mean():.4f}")
    print(f"  category    : row norm {np.linalg.norm(cat_part, axis=1).mean():.4f}")
    assert np.isfinite(feats).all()
    assert np.allclose(np.linalg.norm(text_part, axis=1), 1.0, atol=1e-3)

blair
  text block  : mean +0.00001, row norm 1.0000
  category    : row norm 1.0000
minilm
  text block  : mean -0.00000, row norm 1.0000
  category    : row norm 1.0000
tfidf_svd
  text block  : mean -0.00001, row norm 1.0000
  category    : row norm 1.0000


In [26]:
subset_shapes = []
for level in SUBSET_LEVELS:
    mapping = pq.read_table(SUBSETS_DIR / level / "item_mapping.parquet").to_pandas()
    sliced = ft.slice_for_subset(blair_features, mapping)

    # the row for local index i must be the global row for that item
    probe = rng.choice(len(mapping), 200, replace=False)
    expected = blair_features[mapping.sort_values("item_idx")
                              ["global_item_idx"].to_numpy()[probe]]
    assert np.array_equal(sliced[probe], expected)

    subset_shapes.append({"subset": level, "items": len(mapping),
                          "features": sliced.shape[1],
                          "MB": round(sliced.nbytes / 1e6)})
    print(f"{level:>6}: {sliced.shape}")

pd.DataFrame(subset_shapes)

 small: (105298, 807)
medium: (211842, 807)
  full: (368228, 807)


,subset,items,features,MB
0,small,105298,807,340
1,medium,211842,807,684
2,full,368228,807,1189


In [27]:
del blair_emb, minilm_emb, tfidf_emb
import gc; gc.collect()

22

## 7. Checks

Final validation confirms that the feature matrices are correctly assembled and aligned with each dataset subset. It also verifies that every cold item has a corresponding feature vector, ensuring the cold-start experiments can evaluate all intended items.

In [28]:
checks = []

def check(label, condition):
    checks.append((label, bool(condition)))

check("metadata row count", len(metadata) == EXPECTED_ITEMS)
check("metadata ordered by item_idx",
      (metadata["item_idx"].to_numpy() == np.arange(len(metadata))).all())

for key in LEAKY_DETAIL_KEYS:
    check(f"'{key}' absent from clean_text",
          not metadata["clean_text"].str.contains(key, na=False).any())

for name, feats in feature_blocks.items():
    text_dim = feats.shape[1] - category_onehot.shape[1]
    check(f"{name}: row count", len(feats) == EXPECTED_ITEMS)
    check(f"{name}: finite", np.isfinite(feats).all())
    check(f"{name}: text centred", abs(feats[:, :text_dim].mean()) < 1e-3)
    check(f"{name}: text unit norm",
          np.allclose(np.linalg.norm(feats[:, :text_dim], axis=1), 1.0, atol=1e-3))

check("one category per item", (category_onehot.sum(axis=1) == 1).all())

# subset alignment, all encoders
for level in SUBSET_LEVELS:
    mapping = pq.read_table(SUBSETS_DIR / level / "item_mapping.parquet").to_pandas()
    global_idx = mapping.sort_values("item_idx")["global_item_idx"].to_numpy()
    for name, feats in feature_blocks.items():
        sliced = ft.slice_for_subset(feats, mapping)
        check(f"{level}/{name}: rows match mapping", len(sliced) == len(mapping))
        check(f"{level}/{name}: row order correct",
              np.array_equal(sliced[:500], feats[global_idx[:500]]))

# cold items must have features - this is what the cold-start experiment depends on
for level in SUBSET_LEVELS:
    cold = pq.read_table(
        SUBSETS_DIR / level / "splits" / "timestamp" / "cold_items.parquet"
    ).to_pandas()["item_idx"].to_numpy()
    mapping = pq.read_table(SUBSETS_DIR / level / "item_mapping.parquet").to_pandas()
    sliced = ft.slice_for_subset(feature_blocks["blair"], mapping)
    check(f"{level}: all {len(cold):,} cold items have features",
          len(cold) > 0 and np.isfinite(sliced[cold]).all())

width = max(len(l) for l, _ in checks)
for label, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'}  {label:{width}s}")

assert all(p for _, p in checks), "some feature checks failed"
print(f"\nall {len(checks)} checks passed")

PASS  metadata row count                         
PASS  metadata ordered by item_idx               
PASS  'Best Sellers Rank' absent from clean_text 
PASS  blair: row count                           
PASS  blair: finite                              
PASS  blair: text centred                        
PASS  blair: text unit norm                      
PASS  minilm: row count                          
PASS  minilm: finite                             
PASS  minilm: text centred                       
PASS  minilm: text unit norm                     
PASS  tfidf_svd: row count                       
PASS  tfidf_svd: finite                          
PASS  tfidf_svd: text centred                    
PASS  tfidf_svd: text unit norm                  
PASS  one category per item                      
PASS  small/blair: rows match mapping            
PASS  small/blair: row order correct             
PASS  small/minilm: rows match mapping           
PASS  small/minilm: row order correct            


## 8. Save

The generated embeddings and categorical features are already stored from the previous sections, so only a summary is saved here. A final round-trip check confirms that the feature loader reconstructs the same feature matrices that were validated in this notebook.

In [30]:
for level in SUBSET_LEVELS:
    mapping = pq.read_table(SUBSETS_DIR / level / "item_mapping.parquet").to_pandas()
    for name in feature_blocks:
        expected = ft.slice_for_subset(feature_blocks[name], mapping)
        actual = ft.load_item_features(FEATURES_DIR, SUBSETS_DIR, level, encoder=name)
        assert np.allclose(actual, expected, atol=1e-6), f"{level}/{name} round trip failed"

print(f"load_item_features reproduces all {len(SUBSET_LEVELS) * len(feature_blocks)} "
      f"subset/encoder combinations")

load_item_features reproduces all 9 subset/encoder combinations


In [31]:
summary = {
    "notebook": "07_item_features",
    "items": int(EXPECTED_ITEMS),
    "text_preparation": {
        "source_column": "combined_metadata_text",
        "stripped_keys": LEAKY_DETAIL_KEYS,
        "reason": "crawled in 2023, after the timestamp test period begins",
        "share_before": round(float(before[LEAKY_DETAIL_KEYS[0]]), 4),
        "share_after": round(float(after[LEAKY_DETAIL_KEYS[0]]), 4),
        "max_tokens": MAX_TOKENS,
    },
    "encoders": {
        "blair": {"model": BLAIR_MODEL, "dim": 768, "pooling": "CLS", "role": "primary"},
        "minilm": {"model": MINILM_MODEL, "dim": 384, "pooling": "mean",
                   "max_seq_length": 256, "role": "control"},
        "tfidf_svd": {"components": SVD_COMPONENTS, "role": "control"},
    },
    "categorical": {
        "main_category": {"levels": len(category_levels), "encoding": "one-hot"},
        "store": {"levels": int(metadata.store.nunique()),
                  "encoding": "none - represented via text, too long-tailed to bucket"},
    },
    "feature_dims": {k: int(v.shape[1]) for k, v in feature_blocks.items()},
    "scaling": "text centred then unit-normalised; one-hot already unit norm",
    "storage": "assembled on demand via features.load_item_features()",
}

summary_path = FEATURES_DIR / "features_summary.json"
with open(summary_path, "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, default=str)

print(f"summary -> {summary_path.relative_to(PROJECT_ROOT)}")
print(json.dumps(summary["feature_dims"], indent=2))

summary -> data\processed\features_earliest\features_summary.json
{
  "blair": 807,
  "minilm": 423,
  "tfidf_svd": 295
}


# Findings

- **Temporal leakage was removed.** `Best Sellers Rank`, present in 56.3% of items, was removed before encoding because it reflects product popularity measured after the evaluation period. `Date First Available` was retained as it is known at prediction time.

- **Metadata truncation is unavoidable.** With a 512-token limit, 67.5% of items fit completely. The preserved text prioritises the title, categories, features, and description, while the later details section is truncated when necessary.

- **BLaIR produced meaningful item representations.** Nearest-neighbour inspection returned similar product types, indicating that the embeddings capture useful semantic information.

- **The control encoders produced different representations.** Qualitative examples suggest BLaIR focuses more on product function, while MiniLM and TF-IDF+SVD place greater emphasis on shared brand terms. This will be evaluated quantitatively in the cold-start experiments.

- **`main_category` was included as a categorical feature.** `Store` was not added separately because its high cardinality would largely duplicate information already contained in the text embeddings.

- **Final feature dimensions:** BLaIR 807, MiniLM 423, and TF-IDF+SVD 295 dimensions, including the 39-dimensional `main_category` one-hot encoding.